In [1]:
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# Device setup for M4 Pro
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
import pandas as pd

def load_and_validate_aslg(file_path):
    try:
        # 1. Load the CSV using the default comma separator
        # Using 'utf-8-sig' automatically handles the hidden BOM at the start of the file
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        
        # 2. Strip the embedded newline characters and whitespace from the strings
        df['gloss'] = df['gloss'].str.strip()
        df['text'] = df['text'].str.strip()
        
        # 3. Rename 'text' to 'english' to match your notebook's expected columns
        df.rename(columns={'text': 'english'}, inplace=True)
        
        # 4. Reorder columns so 'english' comes before 'gloss'
        df = df[['english', 'gloss']]
        
        # Drop any empty rows just in case
        df.dropna(inplace=True)
        
        print("--- Dataset Health Report ---")
        print(f"Successfully loaded {len(df)} aligned pairs.\n")
        
        print("--- Alignment Verification (CHECK THIS CAREFULLY) ---")
        print(df.head())
        
        return df

    except Exception as e:
        print(f"Failed to parse data: {e}")
        return None

# Test the function
file_path = "ASLG-PC12_corpus.csv"
df = load_and_validate_aslg(file_path)

--- Dataset Health Report ---
Successfully loaded 87710 aligned pairs.

--- Alignment Verification (CHECK THIS CAREFULLY) ---
                                             english  \
0              ﻿membership of parliament see minutes   
1  approval of minutes of previous sitting see mi...   
2               membership of parliament see minutes   
3            verification of credentials see minutes   
4                     documents received see minutes   

                                          gloss  
0             ﻿MEMBERSHIP PARLIAMENT SEE MINUTE  
1  APPROVAL MINUTE DESC-PREVIOUS SIT SEE MINUTE  
2              MEMBERSHIP PARLIAMENT SEE MINUTE  
3           VERIFICATION CREDENTIALS SEE MINUTE  
4                   DOCUMENT RECEIVE SEE MINUTE  


In [3]:
from datasets import Dataset
from transformers import AutoTokenizer

model_checkpoint = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.src_lang = "eng_Latn" # Tell NLLB the input is standard English

def preprocess_function(examples):
    inputs = [ex for ex in examples["english"]]
    targets = [ex for ex in examples["gloss"]]
    
    # Remove padding="max_length" to allow DataCollator to dynamically pad per-batch
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Convert your Pandas DataFrame into a Hugging Face Dataset for efficient training
raw_dataset = Dataset.from_pandas(df)
tokenized_dataset = raw_dataset.map(preprocess_function, batched=True, remove_columns=raw_dataset.column_names)

# 90/10 Train/Test split for evaluation
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
print(f"Dataset split: {split_dataset}")

Map:   0%|          | 0/87710 [00:00<?, ? examples/s]

Dataset split: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 78939
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8771
    })
})


In [4]:
from transformers import AutoModelForSeq2SeqLM

# Load the model directly onto the M4 Pro GPU (mps) for full fine-tuning
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model.to(device)

# Verify the size of the model
print(f"Total trainable parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Total trainable parameters: 615,073,792


In [5]:
import evaluate
import numpy as np

# Load the ROUGE metric (commonly used for ASL gloss evaluation)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Decode the model's predictions into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels (which are ignored by the loss function) with the pad token ID
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Decode the ground truth labels into text
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Compute the ROUGE score
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    
    # Extract the ROUGE-L score (longest common subsequence)
    return {"rougeL": result["rougeL"]}

In [6]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./asl-gloss-transformer",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4, 
    
    # --- MEMORY OPTIMIZATIONS ---
    optim="adafactor",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    # ----------------------------
    
    weight_decay=1e-3,
    save_total_limit=2,
    num_train_epochs=5, 
    predict_with_generate=True,
    generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
    logging_steps=50
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics
)

# Start training with memory/error handling
try:
    # Clear the MPS cache just in case before starting
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
        
    trainer.train()
    model.save_pretrained("./final_asl_gloss_model")
    print("Training Complete! Model saved to folder.")
except Exception as e:
    print(f"An error occurred during training: {e}")

/opt/anaconda3/envs/asl-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rougel
1,0.062608,0.011636,0.993392
2,0.039185,0.008271,0.995394
3,0.023361,0.007353,0.996228
4,0.010425,0.007202,0.996318
5,0.009056,0.007000,0.996618


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/asl-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/opt/anaconda3/envs/asl-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Complete! Model saved to folder.
